In [2]:
# Author: Arun Bamal
# Project: Data Engineering Assignmentimport duckdb
import pandas as pd
import os

# Create connection
con = duckdb.connect(database='ecommerce.db', read_only=False)

print("Connected to DuckDB")

Connected to DuckDB


In [3]:
df = con.execute("""
SELECT *
FROM read_csv_auto('C:/Users/arunb/Downloads/2019-Oct.csv')
""").fetchdf()

print("Data Loaded:", df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data Loaded: (42448764, 9)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [4]:

df['event_time'] = pd.to_datetime(df['event_time'])

df = df.drop_duplicates()


df['brand'] = df['brand'].fillna('unknown')
df['category_code'] = df['category_code'].fillna('unknown')

print("After cleaning:", df.shape)

After cleaning: (42418544, 9)


In [5]:
# Extract time features
df['event_date'] = df['event_time'].dt.date
df['event_hour'] = df['event_time'].dt.hour
df['day_of_week'] = df['event_time'].dt.day_name()

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,event_date,event_hour,day_of_week
0,2019-10-01 00:00:00,view,44600062,2103807459595387724,unknown,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c,2019-10-01,0,Tuesday
1,2019-10-01 00:00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc,2019-10-01,0,Tuesday
2,2019-10-01 00:00:01,view,17200506,2053013559792632471,furniture.living_room.sofa,unknown,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8,2019-10-01,0,Tuesday
3,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713,2019-10-01,0,Tuesday
4,2019-10-01 00:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,2019-10-01,0,Tuesday


In [6]:

con.execute("""
CREATE OR REPLACE TABLE events AS
SELECT * FROM df
""")

print("Table 'events' created")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Table 'events' created


In [7]:
con.execute("""
CREATE OR REPLACE TABLE daily_sales AS
SELECT 
    event_date,
    COUNT(*) AS total_events,
    SUM(price) AS total_revenue
FROM events
WHERE event_type = 'purchase'
GROUP BY event_date
""")

print("Table 'daily_sales' created")

Table 'daily_sales' created


In [8]:
con.execute("SELECT * FROM daily_sales LIMIT 10").fetchdf()

,event_date,total_events,total_revenue
0,2019-10-01,19305,6275579.06
1,2019-10-02,19469,6213628.53
2,2019-10-03,19255,6233782.98
3,2019-10-04,27039,8623058.19
4,2019-10-05,23492,7341094.46
5,2019-10-06,22169,6737258.17
6,2019-10-07,21378,6348189.06
7,2019-10-08,23071,6819701.26
8,2019-10-09,22747,6855326.05
9,2019-10-10,21992,6665413.21


In [9]:
con.execute("""
SELECT brand, COUNT(*) as cnt
FROM events
GROUP BY brand
ORDER BY cnt DESC
LIMIT 10
""").fetchdf()

,brand,cnt
0,unknown,6112126
1,samsung,5271172
2,apple,4116981
3,xiaomi,3080998
4,huawei,1109758
5,lucente,655818
6,lg,561883
7,bosch,556877
8,oppo,482131
9,sony,456429


In [10]:
con.execute("""
SELECT event_type, COUNT(*) as cnt
FROM events
GROUP BY event_type
""").fetchdf()

,event_type,cnt
0,purchase,742773
1,view,40777328
2,cart,898443


In [11]:
import duckdb
import pandas as pd

# create connection
con = duckdb.connect("ecommerce.db")

print("Connected to DuckDB")

Connected to DuckDB


In [13]:
con.execute("""
CREATE OR REPLACE TABLE staging_events AS
SELECT *
FROM read_csv_auto('C:/Users/arunb/Downloads/2019-Oct.csv')
""")

print("Raw data loaded into staging_events")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw data loaded into staging_events


In [14]:
con.execute("""
SELECT *
FROM staging_events
LIMIT 5
""").fetchdf()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [15]:
con.execute("""
CREATE OR REPLACE TABLE clean_events AS
SELECT
    CAST(event_time AS TIMESTAMP) AS event_time,
    event_type,
    product_id,
    category_id,
    category_code,
    brand,
    CAST(price AS DOUBLE) AS price,
    user_id,
    user_session
FROM staging_events
WHERE price IS NOT NULL
""")

print("Data cleaned and stored in clean_events")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data cleaned and stored in clean_events


In [16]:
con.execute("""
SELECT *
FROM clean_events
LIMIT 5
""").fetchdf()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [17]:
con.execute("""
CREATE OR REPLACE TABLE analytics AS
SELECT
    DATE(event_time) AS event_date,
    event_type,
    COUNT(*) AS total_events,
    SUM(price) AS total_revenue
FROM clean_events
GROUP BY 1, 2
""")

print("Analytics table created")

Analytics table created


In [18]:
con.execute("""
SELECT *
FROM analytics
ORDER BY event_date
LIMIT 10
""").fetchdf()

,event_date,event_type,total_events,total_revenue
0,2019-10-01,cart,16658,6.123620e+06
1,2019-10-01,purchase,19307,6.275964e+06
2,2019-10-01,view,1208280,3.582357e+08
3,2019-10-02,view,1154591,3.450874e+08
4,2019-10-02,cart,17268,6.272537e+06
5,2019-10-02,purchase,19469,6.213629e+06
6,2019-10-03,cart,19323,7.076894e+06
7,2019-10-03,view,1088725,3.261706e+08
8,2019-10-03,purchase,19255,6.233783e+06
9,2019-10-04,view,1346320,3.994617e+08


In [19]:
con.execute("""
COPY analytics TO 'analytics.parquet' (FORMAT PARQUET)
""")

print("Saved as Parquet file")

Saved as Parquet file


In [20]:
import os

print(os.listdir())


['01_schema_design.ipynb', '02_etl_pipeline.ipynb', '03_benchmarks.ipynb', '04_queries.ipynb', '05_visualizations.ipynb', 'analytics.parquet', 'ecommerce.db', 'ecommerce.db.wal']


In [2]:
import duckdb

con = duckdb.connect("ecommerce.db")

con.execute(open("../sql/schema.sql").read())

FileNotFoundError: [Errno 2] No such file or directory: '../sql/schema.sql'

In [3]:
import os
print(os.listdir())


['01_schema_design.ipynb', '02_etl_pipeline.ipynb', '03_benchmarks.ipynb', '04_queries.ipynb', '05_visualizations.ipynb', 'analytics.parquet', 'category_revenue.png', 'ecommerce.db']


In [5]:
import duckdb

con = duckdb.connect("ecommerce.db")

con.execute(open("../sql/schema.sql").read())

In [14]:
# CATEGORY
con.execute("""
INSERT INTO dim_category
SELECT DISTINCT
    category_id,
    category_code,
    split_part(category_code, '.', 1) AS main_category,
    split_part(category_code, '.', 2) AS sub_category
FROM clean_events
WHERE category_id IS NOT NULL
AND category_code IS NOT NULL;
""")

# PRODUCT
con.execute("""
INSERT INTO dim_product
SELECT
    product_id,
    ANY_VALUE(category_id) AS category_id,
    ANY_VALUE(brand) AS brand,
    MAX(price) AS base_price
FROM clean_events
WHERE product_id IS NOT NULL
GROUP BY product_id;
""")

# DATE
con.execute("""
INSERT INTO dim_date
SELECT DISTINCT
    CAST(strftime(event_time, '%Y%m%d%H') AS INTEGER),
    date_trunc('hour', event_time),
    EXTRACT(year FROM event_time),
    EXTRACT(month FROM event_time),
    EXTRACT(day FROM event_time),
    EXTRACT(hour FROM event_time),
    EXTRACT(dow FROM event_time),
    CASE WHEN EXTRACT(dow FROM event_time) IN (0,6) THEN TRUE ELSE FALSE END
FROM clean_events;
""")

# FACT TABLE
con.execute("""
INSERT INTO fact_events (
    date_key,
    event_time,
    event_type,
    product_id,
    user_id,
    user_session,
    price
)
SELECT
    CAST(strftime(event_time, '%Y%m%d%H') AS INTEGER),
    event_time,
    event_type,
    product_id,
    user_id,
    user_session,
    price
FROM clean_events;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ConstraintException: Constraint Error: Violates foreign key constraint because key "category_id: 2070747673266226092" does not exist in the referenced table

In [13]:
con.execute("DELETE FROM dim_category")

In [15]:
import duckdb

con = duckdb.connect("ecommerce.db")

con.execute("DELETE FROM dim_category")
con.execute("DELETE FROM dim_product")
con.execute("DELETE FROM dim_date")
con.execute("DELETE FROM fact_events")

print("Tables cleaned")

Tables cleaned


In [16]:
con.execute("""
INSERT INTO dim_category
SELECT DISTINCT
    category_id,
    category_code,
    COALESCE(split_part(category_code, '.', 1), 'unknown') AS main_category,
    split_part(category_code, '.', 2) AS sub_category
FROM clean_events
WHERE category_id IS NOT NULL
AND category_code IS NOT NULL;
""")

print("dim_category done")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dim_category done


In [18]:
con.execute("DELETE FROM dim_product")

con.execute("""
INSERT INTO dim_product
SELECT
    e.product_id,
    MIN(e.category_id) AS category_id,
    ANY_VALUE(e.brand) AS brand,
    MAX(e.price) AS base_price
FROM clean_events e
JOIN dim_category c
    ON e.category_id = c.category_id
WHERE e.product_id IS NOT NULL
GROUP BY e.product_id;
""")

print("dim_product done")

dim_product done


In [20]:
con.execute("DELETE FROM dim_date")

con.execute("""
INSERT INTO dim_date
SELECT
    date_key,
    MIN(event_time) AS event_hour,
    EXTRACT(year FROM MIN(event_time)),
    EXTRACT(month FROM MIN(event_time)),
    EXTRACT(day FROM MIN(event_time)),
    EXTRACT(hour FROM MIN(event_time)),
    EXTRACT(dow FROM MIN(event_time)),
    CASE 
        WHEN EXTRACT(dow FROM MIN(event_time)) IN (0,6) 
        THEN TRUE ELSE FALSE 
    END
FROM (
    SELECT 
        CAST(strftime(event_time, '%Y%m%d%H') AS INTEGER) AS date_key,
        event_time
    FROM clean_events
) t
GROUP BY date_key;
""")

print("dim_date done")

dim_date done


In [23]:
con.execute("DELETE FROM fact_events")

con.execute("""
INSERT INTO fact_events (
    date_key,
    event_time,
    event_type,
    product_id,
    user_id,
    user_session,
    price
)
SELECT
    CAST(strftime(e.event_time, '%Y%m%d%H') AS INTEGER) AS date_key,
    e.event_time,
    e.event_type,
    e.product_id,
    e.user_id,
    e.user_session,
    e.price
FROM clean_events e

JOIN dim_product p
    ON e.product_id = p.product_id

JOIN dim_date d
    ON CAST(strftime(e.event_time, '%Y%m%d%H') AS INTEGER) = d.date_key

WHERE e.product_id IS NOT NULL
AND e.user_id IS NOT NULL
AND e.user_session IS NOT NULL;   -- ✅ THIS LINE FIXES YOUR ERROR
""")

print("fact_events done")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_events done


In [24]:
print(con.execute("SELECT COUNT(*) FROM fact_events").fetchall())

[(28933153,)]


In [26]:
con.close()